# Diabetic Retinopathy Grading: Colab Runner

**Abdul Aleem Mohammed** | CS 898BA, Wichita State University

Trains the DR grading model end to end. Set the runtime to GPU before running:
`Runtime` -> `Change runtime type` -> `Hardware accelerator: GPU`.

In [ ]:
!nvidia-smi

## 1. Clone the repository and install dependencies

In [ ]:
!git clone https://github.com/thealeemquadri/AbdulAleemMohammed-CS898BA-Project.git
%cd AbdulAleemMohammed-CS898BA-Project
!pip install -q -r requirements.txt
!pip install -q --upgrade kaggle

## 2. Kaggle credentials

1. kaggle.com -> avatar -> Settings -> API -> **Create New Token**. You get a `KGAT_...` string.
2. Open the [APTOS 2019 competition page](https://www.kaggle.com/competitions/aptos2019-blindness-detection)
   and click **Join Competition** to accept the rules, otherwise the download returns 403.

In [ ]:
from getpass import getpass
import os

KAGGLE_TOKEN = getpass("Paste your Kaggle API token (KGAT_...): ")
os.environ["KAGGLE_API_TOKEN"] = KAGGLE_TOKEN

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
token_path = os.path.expanduser("~/.kaggle/access_token")
with open(token_path, "w") as f:
    f.write(KAGGLE_TOKEN)
os.chmod(token_path, 0o600)

KAGGLE_TOKEN = None
print("Kaggle credentials installed.")

## 3. Download the APTOS 2019 dataset

In [ ]:
!mkdir -p data
!kaggle competitions download -c aptos2019-blindness-detection -p data
!unzip -q -o data/aptos2019-blindness-detection.zip -d data
!rm -f data/aptos2019-blindness-detection.zip

import pandas as pd

df = pd.read_csv("data/train.csv")
print("Total images:", len(df))
print("\nClass distribution:")
print(df["diagnosis"].value_counts().sort_index())

## 4. Inspect the processing pipeline on a real fundus image

In [ ]:
import cv2
import matplotlib.pyplot as plt
from src.preprocessing.fundus import preprocess, preprocess_baseline, preprocess_rgb

row = df[df.diagnosis == 3].iloc[0]
img = cv2.cvtColor(cv2.imread(f"data/train_images/{row.id_code}.png"), cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(1, 4, figsize=(20, 5))
ax[0].imshow(img)
ax[0].set_title("Raw fundus (grade 3)")
ax[1].imshow(preprocess_baseline(img))
ax[1].set_title("Baseline: resize only")
ax[2].imshow(preprocess(img))
ax[2].set_title("Full pipeline (gray + morphology)")
ax[3].imshow(preprocess_rgb(img))
ax[3].set_title("RGB pivot (colour preserved)")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

## 5. Train all three configurations

Identical model, seed, and schedule. The only variable is the image processing.
Roughly 35 minutes each on a T4. Reduce `--batch_size` to 8 if you hit CUDA OOM.

In [ ]:
!python -m src.train --config baseline_resize_only --tag baseline --epochs 12 --batch_size 16

In [ ]:
!python -m src.train --config full_pipeline --tag full --epochs 12 --batch_size 16

In [ ]:
!python -m src.train --config rgb_full --tag rgb_full --epochs 12 --batch_size 16

## 6. Compare the runs

In [ ]:
import json

runs = {t: json.load(open(f"outputs/{t}_results.json"))
        for t in ["baseline", "full", "rgb_full"]}

print(f"{'Run':<28}{'QWK':>9}{'Acc':>9}{'dQWK':>10}")
print("-" * 56)
b = runs["baseline"]["final_qwk"]
for name, r in runs.items():
    d = "" if name == "baseline" else f"{r['final_qwk'] - b:+.4f}"
    print(f"{name:<28}{r['final_qwk']:>9.4f}{r['final_accuracy']:>9.4f}{d:>10}")

## 7. Generate the figures

In [ ]:
!python -m src.figures --results outputs/full_results.json --severity 3

from IPython.display import Image, display

for f in ["figures/pipeline_demo.png", "figures/class_dist.png",
          "figures/training_curves.png", "figures/confusion_matrix.png"]:
    display(Image(f))